<center style="padding: 1em 2em;  margin: 1em 0;">
    <h1 style="font-size: 2.2em; margin: 0 0 0.5em 0; color:rgb(52, 72, 92);">
        «Итераторы и генераторы»
    </h1>
    <div style="font-size: 1.4em; color:rgb(52, 72, 92); font-weight: 600;">
        14 октября 2026
    </div>
</center>

Сегодня мы будем брать следующее значение и не собирать всю последовательность заранее.

На прошлой паре `return` уничтожал кадр и отдавал одно значение. `for` по списку мы уже пишем. Сначала посмотрим, из каких вызовов он собран.

---

## **Сегодня пройдемся по этому плану:**

1. **Цикл** `for` – следующий элемент и как цикл понимает, что элементы кончились.
2. **Итерируемое и итератор** – список можно спросить снова, итератор нельзя.
3. `range` – три числа в памяти, не миллион.
4. **Генератор** – функция, чей кадр переживает `yield`.

Каждый кусок нужен следующему.

---

## **Цикл for.**

### **Одного значения мало.**

Вызов создаёт кадр. `return` этот кадр уничтожает. Спросить тот же вызов ещё раз уже нельзя.

In [ ]:
def first_name():
    return "Анна"


print(first_name())  # -> Анна

Имён часто несколько, и держать их все списком незачем: нужно следующее, потом ещё следующее.

`for` такой обход уже показывает. Вопрос, что он вызывает на каждом шаге.

### **Один next.**

Возьмём два имени. На каждом шаге цикл получает один элемент, не весь список.

Сначала у объекта просят итератор. Потом у итератора – следующий элемент.

<div style="padding: 12px 16px; margin: 16px 0; border-radius: 6px; border-left: 4px solid #64748b; background-color: #f8fafc; color: #475569;">
<strong>Итератор:</strong><br>Объект, у которого спрашивают следующий элемент. <code>iter(obj)</code> просит итератор. <code>next(iterator)</code> отдаёт один следующий элемент.
</div>

In [ ]:
names = ["Анна", "Борис"]
iterator = iter(names)

print(next(iterator))  # -> Анна
print(next(iterator))  # -> Борис

`next` забирает элемент и сдвигает итератор. Отдельно посмотреть, не забирая, нельзя.

После Бориса элементы кончились. Ещё один `next` пустую строку не вернёт: он бросит исключение.

<div style="padding: 12px 16px; margin: 16px 0; border-radius: 6px; border-left: 4px solid #64748b; background-color: #f8fafc; color: #475569;">
<strong>StopIteration:</strong><br>Исключение «элементов больше нет».
</div>

`for` делает те же `next` и сам ловит конец. `try` / `except` здесь только эта ловушка: поймали `StopIteration` и вышли. Отдельный разговор про исключения сейчас не нужен.

In [ ]:
names = ["Анна", "Борис"]
iterator = iter(names)

while True:
    try:
        name = next(iterator)
    except StopIteration:
        break
    print(name)
# -> Анна
# -> Борис

Так мы смотрим устройство. В программе остаётся `for`.

<div style="padding: 12px 16px; margin: 16px 0; border-radius: 6px; border-left: 4px solid #64748b; background-color: #f8fafc; color: #475569;">
<strong>Протокол итерации:</strong><br>Три шага, из которых собран <code>for</code>: <code>iter</code>, затем <code>next</code>, пока не придёт <code>StopIteration</code>.
</div>

`next` после конца бросает `StopIteration`. Цикл на этом останавливается. Вызов `next` без цикла ту же ошибку показывает наружу.

---

## **Итерируемое и итератор.**

### **Список можно спросить снова.**

Итератор из прошлого шага помнит, что уже отдал. Список так не устроен: он каждый раз выдаёт новый итератор.

<div style="padding: 12px 16px; margin: 16px 0; border-radius: 6px; border-left: 4px solid #64748b; background-color: #f8fafc; color: #475569;">
<strong>Итерируемое:</strong><br>Объект, у которого <code>iter</code> можно попросить итератор, и попросить снова. Каждый новый итератор начинает с начала. Список – такой объект.
</div>

Два `for` по одному списку оба печатают имена: у каждого цикла свой итератор.

In [ ]:
names = ["Анна", "Борис"]

for name in names:
    print(name)
for name in names:
    print("снова", name)
# оба цикла печатают Анну, потом Бориса

`for` и по готовому итератору вызывает `iter`.

У итератора этот вызов возвращает его самого, поэтому позиция не сбрасывается.

Один `next` уже забрал Анну. Следующий `for` продолжает с Бориса. Ещё один `for` по тому же итератору пустой.

In [ ]:
names = ["Анна", "Борис"]
iterator = iter(names)

print(next(iterator))  # -> Анна
for name in iterator:
    print(name)  # -> Борис
for name in iterator:
    print("пусто", name)  # тело не выполняется

Второй проход молчит, потому что элементы уже отдали.

Список при этом на месте: новый `iter(names)` снова начнёт с Анны.

---

## **Конец обхода.**

Общего флага «элементы кончились» нет. Список сам не помнит, что его уже читали. Каждый итератор держит своё правило.

Итератор списка помнит индекс. На `next` сравнивает его с `len`. Индекс равен длине – бросает `StopIteration`. Иначе отдаёт элемент и сдвигает индекс.

У `range` другое сравнение: текущее число и конец. Какое число уже отдал, какой стоит в `stop`. Как эти три числа лежат в памяти – следующий шаг.

Когда появится функция с `yield`, конец будет другим: конец тела. К этому вернёмся.

Снаружи правило одно. `next` либо отдаёт элемент, либо бросает `StopIteration`. `for` это исключение ловит и выходит.

---

## **range.**

Список из миллиона чисел уже лежит в памяти до первого шага. Часто нужно только следующее число.

`range` хранит начало, конец и шаг. Очередное целое считает итератор, когда его спрашивают.

In [ ]:
numbers = range(10 ** 6)
print(numbers)  # -> range(0, 1000000)
print(numbers.start, numbers.stop, numbers.step)  # -> 0 1000000 1

Если набрать `numbers.` и смотреть поля – там те же `start`, `stop`, `step`. Это поля маленького объекта, не миллион значений.

`list(numbers)` собрал бы миллион чисел сразу. Сам `range` от длины почти не толстеет.

In [ ]:
import sys

print(sys.getsizeof(range(10 ** 6)))      # маленький объект, десятки байт
print(sys.getsizeof(list(range(1000))))   # тысяча чисел уже больше

Числа на вашей машине могут чуть отличаться. Важно другое: миллион в `range` меньше, чем тысяча в списке.

---

## **range не итератор.**

Сам `range` можно спросить снова. Пустым становится итератор, не `range`.

In [ ]:
small = range(3)
print(list(small))  # -> [0, 1, 2]
print(list(small))  # -> [0, 1, 2]

`next` зовут у итератора. У самого `range` следующего числа ещё нет.

In [ ]:
small = range(3)
# next(small)  # TypeError: 'range' object is not an iterator

iterator = iter(small)
print(next(iterator))  # -> 0
print(next(iterator))  # -> 1

<div style="padding: 12px 16px; margin: 16px 0; border-radius: 6px; border-left: 4px solid #64748b; background-color: #f8fafc; color: #475569;">
<strong>__iter__ и __next__:</strong><br><code>__iter__</code> вызывает <code>iter</code>. У списка и у <code>range</code> он возвращает новый итератор. У итератора возвращает сам итератор. <code>__next__</code> вызывает <code>next</code>: следующий элемент или <code>StopIteration</code>.
</div>

`__next__` есть у итератора, который отдаёт `range`, а не у самого `range`. Поэтому `next(range(3))` – `TypeError`.

Свой порядок функцией, без класса, пишем следующим шагом. Классы – отдельная пара.

---

## **Генератор.**

### **yield.**

`range` умеет следующее число, потому что правило простое: начало, конец, шаг.

Обратный отсчёт мы уже писали рекурсией. Там кадры стояли на стеке, пока внешний вызов ждал `return`.

Здесь внешний код значение уже забрал и пошёл дальше. Хочется отдавать по одному числу, не держа все кадры рекурсии на стеке.

На прошлой паре оставался вопрос: можно ли оставить кадр живым и отдавать из него значения по одному.

Можно. Для этого в теле пишут `yield`.

<div style="padding: 12px 16px; margin: 16px 0; border-radius: 6px; border-left: 4px solid #64748b; background-color: #f8fafc; color: #475569;">
<strong>yield:</strong><br>Отдать одно значение и оставить кадр жить. Локальные имена и место в теле сохраняются. Следующий <code>next</code> продолжает тот же кадр со строки после <code>yield</code>.
</div>

`return` на этом месте кадр уничтожает.

На `yield` кадр остаётся цел: локальные имена живы.

Между двумя `next` он не занимает стек поверх вызывающего, поэтому снаружи успевает выполниться `print`.

Где именно он хранится между двумя `next`, скажем на следующем шаге. Сейчас видно другое: после того как снаружи забрали число, тело продолжается, и `n` то же.

In [ ]:
def countdown(n):
    while n > 0:
        yield n
        print("кадр жив, n =", n)
        n = n - 1


steps = countdown(2)
print("взяли", next(steps))
print("взяли", next(steps))
# взяли 2
# кадр жив, n = 2
# взяли 1

Первый `next` остановился на `yield` и отдал `2`. Строка после `yield` выполнилась уже на втором `next`: кадр не умер, `n` всё ещё `2`. Потом цикл дошёл до следующего `yield` и отдал `1`.

`for` по-прежнему своего кадра не создаёт. Кадр, который здесь на паузе, – кадр `countdown`.

### **Функция и объект.**

В прошлом примере вызов мы положили в `steps` и только потом вызвали `next`. Посмотрим, в какой момент вообще начинается тело. Функция и то, что она вернула, – не одно и то же.

<div style="padding: 12px 16px; margin: 16px 0; border-radius: 6px; border-left: 4px solid #64748b; background-color: #f8fafc; color: #475569;">
<strong>Функция-генератор и объект-генератор:</strong><br>Функция-генератор – объект функции, в теле которой есть <code>yield</code>. Вызов этой функции тело не выполняет. Объект-генератор – итератор, который возвращает этот вызов. В нём лежит кадр: ещё не запущенный, а после <code>yield</code> – приостановленный. Первый <code>next</code> выполняет тело до первого <code>yield</code>.
</div>

In [ ]:
def countdown(n):
    print("тело пошло, n =", n)
    yield n
    yield n - 1


print(type(countdown))  # -> <class 'function'>
steps = countdown(3)
print(type(steps))  # -> <class 'generator'>
print("вызов уже вернулся")
print(next(steps))
print(next(steps))
# вызов уже вернулся
# тело пошло, n = 3
# 3
# 2

`countdown` остаётся объектом функции, как на прошлой паре. `steps` – итератор, и кадр лежит в нём. `iter(steps) is steps`: у итератора `iter` возвращает его самого.

Первый `next` дошёл до первого `yield` и остановился: второй `yield` выполнился следующим `next`. Строка «тело пошло» появилась после «вызов уже вернулся». До `next` тело стояло.

Два вызова – два объекта и два кадра. Сдвиг одного второй не трогает.

In [ ]:
first = countdown(3)
second = countdown(3)
print(next(first))
print(next(second))
# тело пошло, n = 3
# 3
# тело пошло, n = 3
# 3

### **return.**

Кадр, который пережил `yield`, всё равно когда-то кончается. Это тот самый конец тела, который мы откладывали: правило «кончено» у генератора – не индекс и не `stop`, а выход из тела.

`return` в этой функции кадр уничтожает. Наружу это выходит как `StopIteration`: тот самый конец, на котором останавливается `for`.

Значение из `return` в результат `next` не попадает. `next` возвращает только то, что стояло в `yield`. Значение `return` лежит у исключения.

In [ ]:
def countdown(n):
    while n > 0:
        yield n
        n = n - 1
    return "конец"


steps = countdown(2)
print(next(steps))  # -> 2
print(next(steps))  # -> 1
try:
    next(steps)
except StopIteration as stop:
    print(stop.value)  # -> конец

`for` это исключение ловит и из цикла выходит. Строку `"конец"` он не печатает: это не элемент обхода.

Если тела после последнего `yield` нет и `return` не написан, конец тот же: `StopIteration`, а значение у него `None`.

### **for по генератору.**

`for` здесь тот же: `iter`, потом `next`, пока не придёт `StopIteration`. `iter` вернёт сам объект-генератор, позиция не сбросится.

In [ ]:
def countdown(n):
    while n > 0:
        yield n
        n = n - 1


for n in countdown(3):
    print(n)
# -> 3
# -> 2
# -> 1

Повторный `for` по тому же объекту пустой. Нужен новый вызов `countdown(3)` – новый кадр.

`list(countdown(3))` тоже заберёт все значения, но соберёт их в список. Для показа удобно. Для длинного обхода обычно хватает самого `for`.

### **Выражение-генератор.**

Функция на одно действие – «следующий квадрат» – длинновата. Тот же объект-генератор записывают выражением.

<div style="padding: 12px 16px; margin: 16px 0; border-radius: 6px; border-left: 4px solid #64748b; background-color: #f8fafc; color: #475569;">
<strong>Выражение-генератор:</strong><br>Объект-генератор, записанный круглыми скобками. На каждый элемент один <code>yield</code>, как в функции выше. Выражение создаёт объект сразу. Очередной квадрат считает тот <code>next</code>, который его спросил.
</div>

Квадратные скобки с `for` внутри собирают весь список в этой строке. Круглые скобки список не строят.

In [ ]:
squares_now = [n * n for n in range(4)]
print(squares_now)  # -> [0, 1, 4, 9]

squares = (n * n for n in range(4))
print(next(squares))  # -> 0
print(next(squares))  # -> 1

В `squares_now` четыре числа уже лежат. В `squares` посчитаны только те, которые спросили: `0` и `1`. `4` и `9` ещё нет. Это тот же одноразовый итератор: когда он отдаст и их, следующий проход будет пустой. Список `squares_now` можно обойти снова.

### **Бесконечный обход.**

Пока каждый `next` только продолжает кадр, хранить будущие элементы негде и не нужно. Генератор может не заканчиваться: в кадре одно имя `n`.

In [ ]:
def naturals():
    n = 0
    while True:
        yield n
        n = n + 1


nums = naturals()
print(next(nums))  # -> 0
print(next(nums))  # -> 1
print(next(nums))  # -> 2

`for` без выхода отсюда не получит `StopIteration` и не остановится. Берём несколько `next` или выходим сами.

In [ ]:
for n in naturals():
    if n == 3:
        break
    print(n)
# -> 0
# -> 1
# -> 2

`list(naturals())` сюда не подходит: списка конца нет.

### **Два генератора подряд.**

Один проход может отфильтровать, второй – посчитать. Если оба собирают список, в памяти сначала лежат все чётные, потом все квадраты.

Если оба отдают по одному числу, в памяти только то, что сейчас спросили.

In [ ]:
def even(numbers):
    for n in numbers:
        if n % 2 == 0:
            yield n


def squares(numbers):
    for n in numbers:
        yield n * n


print(sum(squares(even(range(10)))))  # -> 120

`even` отдаёт следующее чётное, `squares` сразу возводит его в квадрат, `sum` забирает это число. Промежуточного списка нет: в памяти одно целое из `range`, одно из `even`, одно из `squares`.

Тот же конвейер круглыми скобками: `(n * n for n in numbers if n % 2 == 0)`. Функция нужна, когда шагов несколько или хочется имя.

На следующей паре тронем приостановленный кадр снаружи: положим значение, бросим исключение, закроем. Потом – готовый обход в `itertools`. Потом файл: сначала куда уходит `print`, затем тот же `next`, элемент – строка с диска.